In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
spark = SparkSession.builder.appName("Expense_ETL").getOrCreate()

In [0]:
# ==========================================
# STEP 1 : CREATE USERS DATA
# ==========================================

users_data = [
    (101,"Siva Kumar","siva@gmail.com"),
    (102,"Ravi Kumar","ravi@gmail.com"),
    (103,"Anita Sharma","anita@gmail.com"),
    (104,"John Peter","john@gmail.com")
]

users_df = spark.createDataFrame(
    users_data,
    ["user_id","full_name","email"]
)

display(users_df)

# Save Users CSV

users_df.coalesce(1).write \
.mode("overwrite") \
.option("header",True) \
.csv("/FileStore/tables/users")

user_id,full_name,email
101,Siva Kumar,siva@gmail.com
102,Ravi Kumar,ravi@gmail.com
103,Anita Sharma,anita@gmail.com
104,John Peter,john@gmail.com


In [0]:
expenses_data = [

(1,101,"2026-06-01","Food",500),

(2,101,"2026-06-10","Shopping",700),

(3,102,"2026-06-05","Bills",1000),

(4,103,"2026-06-03","Food",400),

(5,101,"2026-07-02","Travel",12000),

(6,102,"2026-07-04","Food",900),

(7,104,"2026-07-08","Bills",1500),

(8,104,"2026-07-10","Travel",20000)

]
expenses_df = spark.createDataFrame(

expenses_data,

["expense_id","user_id","date","category","amount"]

)

display(expenses_df)

expenses_df.coalesce(1).write \
.mode("overwrite") \
.option("header",True) \
.csv("/FileStore/tables/expenses")

expense_id,user_id,date,category,amount
1,101,2026-06-01,Food,500
2,101,2026-06-10,Shopping,700
3,102,2026-06-05,Bills,1000
4,103,2026-06-03,Food,400
5,101,2026-07-02,Travel,12000
6,102,2026-07-04,Food,900
7,104,2026-07-08,Bills,1500
8,104,2026-07-10,Travel,20000


In [0]:

users = spark.read \
.option("header",True) \
.option("inferSchema",True) \
.csv("/FileStore/tables/users")

expenses = spark.read \
.option("header",True) \
.option("inferSchema",True) \
.csv("/FileStore/tables/expenses")

display(users)

display(expenses)

user_id,full_name,email
101,Siva Kumar,siva@gmail.com
102,Ravi Kumar,ravi@gmail.com
103,Anita Sharma,anita@gmail.com
104,John Peter,john@gmail.com


expense_id,user_id,date,category,amount
1,101,2026-06-01,Food,500
2,101,2026-06-10,Shopping,700
3,102,2026-06-05,Bills,1000
4,103,2026-06-03,Food,400
5,101,2026-07-02,Travel,12000
6,102,2026-07-04,Food,900
7,104,2026-07-08,Bills,1500
8,104,2026-07-10,Travel,20000


In [0]:
expenses = expenses.withColumn(
    "amount",
    col("amount").cast("double")
)

expenses = expenses.withColumn(
    "date",
    to_date("date")
)

expenses = expenses.dropna()

display(expenses)

expense_id,user_id,date,category,amount
1,101,2026-06-01,Food,500.0
2,101,2026-06-10,Shopping,700.0
3,102,2026-06-05,Bills,1000.0
4,103,2026-06-03,Food,400.0
5,101,2026-07-02,Travel,12000.0
6,102,2026-07-04,Food,900.0
7,104,2026-07-08,Bills,1500.0
8,104,2026-07-10,Travel,20000.0


In [0]:
joined = users.join(
    expenses,
    "user_id"
)

display(joined)

user_id,full_name,email,expense_id,date,category,amount
101,Siva Kumar,siva@gmail.com,1,2026-06-01,Food,500.0
101,Siva Kumar,siva@gmail.com,2,2026-06-10,Shopping,700.0
102,Ravi Kumar,ravi@gmail.com,3,2026-06-05,Bills,1000.0
103,Anita Sharma,anita@gmail.com,4,2026-06-03,Food,400.0
101,Siva Kumar,siva@gmail.com,5,2026-07-02,Travel,12000.0
102,Ravi Kumar,ravi@gmail.com,6,2026-07-04,Food,900.0
104,John Peter,john@gmail.com,7,2026-07-08,Bills,1500.0
104,John Peter,john@gmail.com,8,2026-07-10,Travel,20000.0


In [0]:

joined = joined.withColumn(
    "month",
    month("date")
)

joined = joined.withColumn(
    "year",
    year("date")
)

display(joined)

user_id,full_name,email,expense_id,date,category,amount,month,year
101,Siva Kumar,siva@gmail.com,1,2026-06-01,Food,500.0,6,2026
101,Siva Kumar,siva@gmail.com,2,2026-06-10,Shopping,700.0,6,2026
102,Ravi Kumar,ravi@gmail.com,3,2026-06-05,Bills,1000.0,6,2026
103,Anita Sharma,anita@gmail.com,4,2026-06-03,Food,400.0,6,2026
101,Siva Kumar,siva@gmail.com,5,2026-07-02,Travel,12000.0,7,2026
102,Ravi Kumar,ravi@gmail.com,6,2026-07-04,Food,900.0,7,2026
104,John Peter,john@gmail.com,7,2026-07-08,Bills,1500.0,7,2026
104,John Peter,john@gmail.com,8,2026-07-10,Travel,20000.0,7,2026


In [0]:
summary = joined.groupBy(
    "user_id",
    "full_name",
    "year",
    "month"
).agg(
    sum("amount").alias("monthly_spend")
)

display(summary)

user_id,full_name,year,month,monthly_spend
103,Anita Sharma,2026,6,400.0
101,Siva Kumar,2026,7,12000.0
102,Ravi Kumar,2026,7,900.0
101,Siva Kumar,2026,6,1200.0
102,Ravi Kumar,2026,6,1000.0
104,John Peter,2026,7,21500.0


In [0]:

summary = summary.withColumn(
    "budget",
    lit(20000)
)

summary = summary.withColumn(
    "savings",
    col("budget")-col("monthly_spend")
)

display(summary)

user_id,full_name,year,month,monthly_spend,budget,savings
103,Anita Sharma,2026,6,400.0,20000,19600.0
101,Siva Kumar,2026,7,12000.0,20000,8000.0
102,Ravi Kumar,2026,7,900.0,20000,19100.0
101,Siva Kumar,2026,6,1200.0,20000,18800.0
102,Ravi Kumar,2026,6,1000.0,20000,19000.0
104,John Peter,2026,7,21500.0,20000,-1500.0


In [0]:
summary = summary.withColumn(
    "alert",
    when(col("monthly_spend")>18000,"High Spending")
    .when(col("monthly_spend")>10000,"Medium Spending")
    .otherwise("Normal")
)

display(summary)

user_id,full_name,year,month,monthly_spend,budget,savings,alert
103,Anita Sharma,2026,6,400.0,20000,19600.0,Normal
101,Siva Kumar,2026,7,12000.0,20000,8000.0,Medium Spending
102,Ravi Kumar,2026,7,900.0,20000,19100.0,Normal
101,Siva Kumar,2026,6,1200.0,20000,18800.0,Normal
102,Ravi Kumar,2026,6,1000.0,20000,19000.0,Normal
104,John Peter,2026,7,21500.0,20000,-1500.0,High Spending


In [0]:
summary.write \
.mode("overwrite") \
.format("delta") \
.save("/FileStore/delta/monthly_summary")

In [0]:
summary.coalesce(1).write \
.mode("overwrite") \
.option("header",True) \
.csv("/FileStore/output/monthly_summary_csv")

In [0]:
result = spark.read.format("delta").load(
"/FileStore/delta/monthly_summary"
)
display(result)

user_id,full_name,year,month,monthly_spend,budget,savings,alert
103,Anita Sharma,2026,6,400.0,20000,19600.0,Normal
101,Siva Kumar,2026,7,12000.0,20000,8000.0,Medium Spending
102,Ravi Kumar,2026,7,900.0,20000,19100.0,Normal
101,Siva Kumar,2026,6,1200.0,20000,18800.0,Normal
102,Ravi Kumar,2026,6,1000.0,20000,19000.0,Normal
104,John Peter,2026,7,21500.0,20000,-1500.0,High Spending


In [0]:
print("Final Monthly Expense Report")
result.show(truncate=False)

Final Monthly Expense Report
+-------+------------+----+-----+-------------+------+-------+---------------+
|user_id|full_name   |year|month|monthly_spend|budget|savings|alert          |
+-------+------------+----+-----+-------------+------+-------+---------------+
|103    |Anita Sharma|2026|6    |400.0        |20000 |19600.0|Normal         |
|101    |Siva Kumar  |2026|7    |12000.0      |20000 |8000.0 |Medium Spending|
|102    |Ravi Kumar  |2026|7    |900.0        |20000 |19100.0|Normal         |
|101    |Siva Kumar  |2026|6    |1200.0       |20000 |18800.0|Normal         |
|102    |Ravi Kumar  |2026|6    |1000.0       |20000 |19000.0|Normal         |
|104    |John Peter  |2026|7    |21500.0      |20000 |-1500.0|High Spending  |
+-------+------------+----+-----+-------------+------+-------+---------------+

